# 02 Group and rank
Group actionable reports, train the urgency model on other events, compare baselines, and inspect the SVI tradeoff. If Stage 1 real outputs are absent, this notebook runs a clearly labeled synthetic fixture rehearsal in the stub output directory. Fixture results are not disaster performance measurements.

In [ ]:
# Bootstrap: make the repo importable whether the notebook runs in IBM's container or ours
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from surgedesk.envcheck import run
ENV = run()

In [ ]:
import random
import pandas as pd
from surgedesk import config, fileio, rank, fairness, baselines
from surgedesk.contracts import Report, Incident, FEATURE_PLAIN, feature_reason
from surgedesk.group import group_event, load_stage1
from surgedesk.textutil import mask_privacy

fixture_mode = not (config.OUTPUTS / "events.json").exists()
if fixture_mode:
    print("SYNTHETIC FIXTURE REHEARSAL: results are not real-event measurements")
    event = config.FIXTURE_EVENT
    train_events = [config.FIXTURE_TRAIN_EVENT]
    source_root = config.FIXTURES / "outputs"
    work_root = config.STUB_OUTPUTS / "agent2"
    values = {i.county_fips: i.svi for i in fileio.read_jsonl(source_root / event / config.INCIDENTS_FILE, Incident)}
    lookup = values.get
    model_path = work_root / config.RANKER_PATH.name
    meta = rank.train_ranker(train_events, svi_lookup=lookup, outputs_root=source_root,
                             ranker_path=model_path, allow_fixture=True)
else:
    events = fileio.read_json(config.OUTPUTS / "events.json")
    train_events = config.train_events(events)
    event, source_root, work_root = config.DEMO_EVENT, config.OUTPUTS, config.OUTPUTS
    lookup = rank.default_svi_lookup()
    model_path = config.RANKER_PATH
    meta = rank.train_ranker(config.train_events(events), svi_lookup=lookup)
display(pd.DataFrame([{k: meta[k] for k in ("n_rows", "pos_rate", "dev_auc", "dev_ap")}]))
print("Ranking objective:", meta["objective"])
print(meta["target_definition"])
print(meta["score_interpretation"])


In [ ]:
incidents = rank.rank_event(event, force=True, svi_lookup=lookup,
                            src_dir=source_root / event, out_dir=work_root / event,
                            ranker=rank.Ranker(model_path))
display(fileio.read_json(work_root / event / "group_stats.json"))

In [ ]:
snap = rank.pick_demo_snapshot(incidents)
selected = [i for i in incidents if i.snapshot_time == snap]
print("Snapshot:", snap)
display(pd.DataFrame([dict(rank=i.rank, place=mask_privacy(i.place or "Unknown"),
                          county=i.county_fips, svi=i.svi, n_reports=i.features["n_reports"],
                          score=i.score, reasons=mask_privacy(
                              "; ".join(f"{feature_reason(k, i.features)} ({v:+.6f})" for k, v in i.shap_top[:3])))
                      for i in selected[:config.TOP_K_EVAL]]))

In [ ]:
reports, tags, embeddings = load_stage1(event, source_root / event)
texts = {r.report_id: r.text for r in reports}
orders = {"SurgeDesk": baselines.order_surgedesk(selected), "FIFO": baselines.order_fifo(selected),
          "Volume": baselines.order_volume(selected), "Keyword": baselines.order_keyword(selected, texts)}
display(pd.DataFrame({name: pd.Series(ids[:config.TOP_K_EVAL]) for name, ids in orders.items()}))

In [ ]:
display(fairness.fairness_table(selected))

## Grouping spot check
Read the masked member texts and judge whether each group describes one incident. Any threshold proposal goes to Person 1; this notebook does not change thresholds. The fixture sample may contain fewer multi-report incidents than the configured spot-check sample size.

In [ ]:
check_event = train_events[0]
check_state = group_event(check_event, src_dir=source_root / check_event, out_dir=work_root / check_event)
multiple = [i for i in check_state.incidents.values() if len(i.reports) > 1]
sample = random.Random(config.SEED).sample(multiple, min(config.PURITY_SAMPLE_N, len(multiple)))
print(f"Spot-check sample: {len(sample)} of {len(multiple)} multi-report incidents from {check_event}")
for incident in sample:
    print("\n" + incident.incident_id)
    for report in incident.reports:
        print(mask_privacy(report.text))